# q_N 조건부 3차원 V 가격위치 표현 + M4: Dunnhumby seed 42

역사적 개발구간(684~690일)에서 동일 실행의 M1, M4-only, 기존 스칼라 N/V M5, 수정 N/V M5, degree 내부 N/V 순열 M5를 각각 100 epoch 학습합니다. 수정 M2는 q_N을 경제표현의 제한된 사용자 게이트로만 사용하고, q_V와 상품 가격 백분위를 같은 저·중·고 3차원 고정 기저에 놓습니다. 아이템 반복구매축은 사용하지 않습니다.

이 노트북은 단일 개발 seed 방향성 확인용입니다. final test와 holdout은 만들지 않으며, 결과로 유의성이나 일반화를 주장하지 않습니다. 중단 뒤 같은 셀을 다시 실행하면 저장된 epoch부터 자동 재개합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, os, shutil, subprocess, sys

REVIEWED_SHA = '487a8a1451d53b009bcca0080666df36db8ceeca'
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
os.chdir('/content')
repo = Path('/content/clv-m2-lightgcn-runner')
clone_errors = []
for clone_attempt in range(1, 4):
    os.chdir('/content')
    if repo.exists():
        shutil.rmtree(repo)
    result = subprocess.run(
        ['git', 'clone', REPO_URL, str(repo)],
        text=True, capture_output=True,
    )
    if result.returncode == 0:
        break
    clone_errors.append(result.stderr.strip())
    print(f'GitHub clone {clone_attempt}/3 실패:', result.stderr.strip())
else:
    raise RuntimeError('GitHub clone 3회 실패:\n' + '\n'.join(clone_errors))
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(
    ['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
for module_name in tuple(sys.modules):
    if module_name.startswith(('lightgcn_', 'clv_')):
        del sys.modules[module_name]
importlib.invalidate_caches()
%cd /content/clv-m2-lightgcn-runner
print('실행 코드 고정 완료:', actual_sha)

In [ ]:
import json
import torch
from lightgcn_clv_m5_n_conditioned_value_basis_screen import (
    MODEL_IDS,
    configure_n_conditioned_value_basis_screen,
    preflight_summary,
    run_n_conditioned_value_basis_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_n_conditioned_value_basis_screen(
    out_dir='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m5_n_conditioned_value_basis_development_screen_v1',
)
summary = preflight_summary(cfg)
assert summary['split'] == 'historical_development_days_684_690'
assert summary['trained_models'] == list(MODEL_IDS)
assert summary['m2']['economic_dim'] == 3
assert summary['m2']['rho'] == 0.05
assert summary['m2']['item_n_or_item_clv_input'] is False
assert summary['m2']['economic_graph_propagation'] is True
assert summary['m4']['actual_assignment_in_all_weighted_arms'] is True
assert summary['fixed']['new_item_task'] is True
assert summary['fixed']['min_item_interactions'] == 1
assert summary['fixed']['final_test_constructed'] is False
assert summary['fixed']['holdout_constructed'] is False
assert summary['fixed']['one_training_loop_and_optimizer'] is True
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_n_conditioned_value_basis_screen(cfg)

In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

print('1) 동일 실행 M1·M4·스칼라 M5·수정 M5·N/V 순열 절대지표')
show(result_df)
print('2) 대조군별 전체 지표 비교')
show(result_df.attrs['comparison'])
print('3) ID 점수 대비 경제점수 영향력')
show(result_df.attrs['score_diagnostics'])
print('4) 수정 M5의 Top-10 변화')
show(result_df.attrs['top10_overlap'])
print('5) 사전 고정 판독')
print(json.dumps(result_df.attrs['decision'], ensure_ascii=False, indent=2))
print('6) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))